# 10 Timestep Gate Sweep

`t_min` / `t_max` 범위를 다르게 설정하고 FID와 linear probe acc를 비교합니다.

In [ ]:
import torch
from src.experiments import load_cfg, deep_update

# ── GPU 자동 감지 ───────────────────────────────────────────────
n_gpu = torch.cuda.device_count()
print(f"감지된 GPU 수: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory // 1024**3} GB")
if n_gpu == 0:
    print("  (GPU 없음 — CPU로 실행됩니다)")


In [ ]:
from src.experiments import load_cfg, deep_update, launch_train
import pandas as pd
from pathlib import Path

base_cfg = load_cfg("configs/cifar10.yaml")

sweep = [
    (0.05, 0.45),
    (0.10, 0.60),
    (0.15, 0.70),
    (0.20, 0.80),
]

rows = []
for t_min, t_max in sweep:
    cfg = deep_update(base_cfg, {"sdd": {"gating": {"t_min": t_min, "t_max": t_max}}})
    cfg["run_name"] = f"cifar10_gate_{t_min:.2f}_{t_max:.2f}"
    print(f"\n▶ t_min={t_min}, t_max={t_max}")
    launch_train(cfg, num_processes=None, with_eval=True)

    csv = Path(f"outputs/logs/{cfg['run_name']}_history.csv")
    if csv.exists():
        df = pd.read_csv(csv)
        last = df.dropna(subset=["val_fid"]).iloc[-1] if "val_fid" in df.columns else df.iloc[-1]
        rows.append({"t_min": t_min, "t_max": t_max,
                     "fid": last.get("val_fid"),
                     "linear_probe_acc": last.get("val_linear_probe_acc")})

results = pd.DataFrame(rows)
results


In [ ]:
import matplotlib.pyplot as plt

if not results.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    x_labels = [f"{r['t_min']:.2f}–{r['t_max']:.2f}" for _, r in results.iterrows()]
    axes[0].bar(x_labels, results["fid"],             color="steelblue")
    axes[0].set_title("FID ↓"); axes[0].set_xlabel("gate range")
    axes[1].bar(x_labels, results["linear_probe_acc"], color="darkorange")
    axes[1].set_title("Linear probe acc ↑"); axes[1].set_xlabel("gate range")
    plt.tight_layout()
    plt.savefig("outputs/figures/timestep_sweep.png", dpi=150)
    plt.show()
